In [ ]:
import anthropic
print(anthropic.__version__)

In [ ]:
from dotenv import load_dotenv
import os

found = load_dotenv("../.env")
key = os.environ.get("ANTHROPIC_API_KEY")
print("файл .env найден:", found)
print("ключ начинается с:", key[:7])

In [ ]:
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Одним предложением: что такое PubMed?"}],
)
print(response.content[0].text)


In [ ]:
print(response.usage)

In [ ]:
input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens
price_input_per_million = 1.0    # $ за 1 000 000 токенов входа (Haiku 4.5)
price_output_per_million = 5.0   # $ за 1 000 000 токенов выхода
cost_usd = ((input_tokens*price_input_per_million) + (output_tokens*price_output_per_million))/1000000
print(f"Цена: ${cost_usd:.6f}")

In [ ]:
def cost_usd(usage):
    input_tokens = usage.input_tokens
    output_tokens = usage.output_tokens
    price_input_per_million = 1.0    # $ за 1 000 000 токенов входа (Haiku 4.5)
    price_output_per_million = 5.0   # $ за 1 000 000 токенов выхода
    return ((input_tokens*price_input_per_million) + (output_tokens*price_output_per_million))/1000000
print(f"Цена: ${cost_usd(response.usage):.6f}")


In [ ]:
import base64
from pathlib import Path

image_path = sorted(Path("../tests/slides/real").glob("*.png"))[0]
image_bytes = image_path.read_bytes()
image_data = base64.standard_b64encode(image_bytes).decode("utf-8")

print(image_path.name, "·", len(image_bytes) // 1024, "КБ")
print("в виде текста:", len(image_data), "символов, начало:", image_data[:40])

In [ ]:
response3 = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=2048,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_data}},
            {"type": "text", "text": "Перечисли по пунктам, что написано на слайде: заголовок, препараты, "
                                     "названия исследований, все числа как на слайде. Ничего не добавляй от себя."},
        ],
    }],
)
print(response3.content[0].text)
print(f"\nвход {response3.usage.input_tokens} ток. · выход {response3.usage.output_tokens} ток. · ${cost_usd(response3.usage):.6f}")